# 4-2 · 조건을 맞춰도 절반인가 — 회귀로 구한 배수와 계획

**⏱ 60분** · 4주차 교재 2/2 — (4-1) 평균으로 비교하기 → 회귀로 조건을 맞춰 비교하고 계획 배수 정하기

## 🎯 이번 시간의 질문

4-1에서 비·눈 시간의 대여는 평시의 **약 0.57배**였습니다. 그런데 비·눈 시간은 평시보다 약 2도 춥고 바람이 셌고, 대여가 많은 여름에는 비·눈이 적게 왔습니다(4-1 과제 2).

> **기온·바람·계절 같은 다른 조건을 맞추면 비·눈의 배수는 얼마인가요? 계획에는 몇 배를 쓸까요?**

4-1 5절처럼 표를 나눠 맞추면 조건이 여러 개일 때 칸이 너무 잘게 쪼개집니다. 여러 조건을 한꺼번에 맞추는 도구가 **회귀분석**입니다. 회귀계수는 식에 함께 넣은 다른 조건이 같을 때의 차이입니다. 대여 건수에 로그를 씌워 회귀하면 이 차이를 **배수**로 읽을 수 있습니다.

이 자료는 날씨를 일부러 바꿔 본 실험이 아니라 운영 중에 쌓인 기록입니다. 조건을 맞춘 배수도 식에 넣은 조건 안에서의 비교입니다.

이번 시간에 필요한 이론(회귀계수의 뜻, 범주 변수, 로그와 배수, 넣지 않을 조건, 날짜로 묶은 신뢰구간)은 **[4주차 배경지식 — 조건을 맞춘 비교](https://balab-pknu.github.io/bizanalytics/theory/week04.html)** 문서에 있습니다.

**이번 시간에 반복할 패턴:** 회귀 두 줄 — `smf.ols("결과 ~ 조건", data=표).fit()` → `np.exp(모형.params["rain"])`

셀 표시는 4-1과 같습니다: ▶ 그대로 실행 · ✍️ 보고 타이핑 · 📝 과제 · 💬 결과 해설 · ✅ 체크포인트

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

In [3]:
# ▶ 준비 셀 — 그대로 실행하세요. 기록을 읽고 4-1에서 만든 두 열(rain, temp_c)을 다시 만듭니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
%matplotlib inline

import os
import sys
for _p in (".", "..", "../.."):
    if os.path.exists(os.path.join(_p, "balab.py")):
        sys.path.insert(0, _p); break


else:
    !wget -q https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py

from balab import load
h = load("bike")
h["rain"] = (h["weathersit"] >= 3).astype(int)
h["temp_c"] = h["temp"] * 47 - 8
print("준비 완료!", h.shape)

준비 완료! (17379, 19)


✅ `준비 완료! (17379, 19)`가 나오면 됩니다. 원래 17개 열에 `rain`과 `temp_c`가 붙었습니다.

---
## 1. 로그의 차이는 배수다

4-1에서 비·눈의 차이는 건수보다 배수로 비슷했습니다. 그런데 회귀식은 더하고 빼는 식입니다. 곱하기를 빼기로 바꿔 주는 **로그**를 쓰면 배수를 회귀식에 담을 수 있습니다.

100건이 50건으로, 400건이 200건으로 줄어든 두 경우의 로그 차이를 계산해 봅니다. `np.log`가 로그입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
print(np.log(50) - np.log(100))
print(np.log(200) - np.log(400))
```

In [4]:
# ✍️ 위 코드를 여기에 직접 입력하세요
print(np.log(50) - np.log(100))
print(np.log(200) - np.log(400))

-0.6931471805599458
-0.6931471805599454


💬 두 값 모두 **−0.693**입니다. 50건이 줄었든 200건이 줄었든 "절반이 되었다"는 사실이 로그 차이 하나로 같게 나타납니다.

`np.exp`는 로그를 되돌립니다. 로그 차이에 `np.exp`를 씌우면 배수가 나옵니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
np.exp(np.log(50) - np.log(100))
```

In [5]:
# ✍️ 위 코드를 여기에 직접 입력하세요
np.exp(np.log(50) - np.log(100))

np.float64(0.4999999999999997)

💬 **0.5배**입니다. 결과에 로그를 씌워 회귀하면 계수가 이런 로그 차이이고, **계수에 `np.exp`를 씌운 값이 배수**입니다.

---
## 2. 작은 예로 보는 "조건을 맞춘다"

회귀가 조건을 어떻게 맞추는지, 손으로 따라갈 수 있는 여섯 시간짜리 기록으로 먼저 봅니다. 새벽 4시와 오후 5시의 기록이고, **어느 시간대든 비가 오면 대여가 평시의 절반**입니다. 다만 비는 새벽에 두 번, 저녁에 한 번 왔습니다.

In [7]:
# ▶ 그대로 실행하세요 — 조건 맞추기 작은 예 toy
toy = pd.DataFrame({"hour": [4, 4, 4, 17, 17, 17],
                    "rain": [0, 1, 1, 0, 0, 1],
                    "rentals": [100, 50, 50, 400, 400, 200]})
toy

,hour,rain,rentals
0,4,0,100
1,4,1,50
2,4,1,50
3,17,0,400
4,17,0,400
5,17,1,200


**회귀 두 줄** — 이번 시간 내내 반복할 패턴입니다.

```python
model = smf.ols("결과 ~ 조건1 + 조건2", data=표).fit()   # 1) 식과 데이터를 주고 적합한다
np.exp(model.params["rain"])                             # 2) rain의 계수 → 배수
```

- 물결표(`~`) 왼쪽에 결과, 오른쪽에 조건을 씁니다. 결과에 로그를 씌우려면 `np.log(rentals)`처럼 씁니다.
- `.fit()`이 최소제곱으로 계수를 구하고, `.params["rain"]`이 rain의 계수입니다.

먼저 조건 없이 비만 넣습니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
toy_m0 = smf.ols("np.log(rentals) ~ rain", data=toy).fit()
np.exp(toy_m0.params["rain"])
```

In [8]:
# ✍️ 위 코드를 여기에 직접 입력하세요
toy_m0 = smf.ols("np.log(rentals) ~ rain", data=toy).fit()
np.exp(toy_m0.params["rain"])

np.float64(0.31498026247371785)

💬 **결과 해설** — **약 0.31배**입니다. 실제로는 어느 시간대든 절반(0.5배)이었는데 더 작게 나왔습니다.
- 비가 대여가 적은 새벽에 두 번 와서, 비 오는 시간 쪽에 새벽 기록이 많이 모였기 때문입니다.
- 날씨의 차이에 **시간대의 차이가 섞인** 것입니다.

이번에는 시간대를 식에 넣습니다. `hour`는 4와 17이라는 숫자지만 크기가 아니라 구분을 나타내므로 `C(hour)`로 감싸 **범주**로 넣습니다. 범주로 넣으면 시간대마다 대여 수준을 따로 둡니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
toy_m1 = smf.ols("np.log(rentals) ~ rain + C(hour)", data=toy).fit()
np.exp(toy_m1.params["rain"])
```

In [9]:
# ✍️ 위 코드를 여기에 직접 입력하세요
toy_m1 = smf.ols("np.log(rentals) ~ rain + C(hour)", data=toy).fit()
np.exp(toy_m1.params["rain"])

np.float64(0.4999999999999993)

💬 **결과 해설** — **0.5배**입니다. 새벽 안에서 50 ÷ 100, 저녁 안에서 200 ÷ 400이 모두 0.5이니 손으로 계산한 값과 같습니다.

**조건을 식에 넣는다 = 그 조건이 같은 기록끼리 비교한다**는 뜻입니다.

---
## 3. 실제 기록에서 조건을 하나씩 더하기

같은 두 줄을 실제 기록 `h`에 씁니다. 조건을 하나씩 더하면서 배수가 어떻게 움직이는지 봅니다.

### 비·눈만 — 출발점

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
m0 = smf.ols("np.log(cnt) ~ rain", data=h).fit()
np.exp(m0.params["rain"])
```

In [10]:
# ✍️ 위 코드를 여기에 직접 입력하세요
m0 = smf.ols("np.log(cnt) ~ rain", data=h).fit()
np.exp(m0.params["rain"])

np.float64(0.5101535707856858)

💬 **결과 해설** — **약 0.51배**입니다. 4-1의 평균끼리 0.57배와 조금 다릅니다.
- 평균끼리 나눌 때는 건수를 그대로 평균했으므로 대여가 수백 건인 저녁 시간이 평균을 크게 좌우합니다.
- 로그를 씌우면 비율의 차이가 평균되므로 대여가 적은 시간도 많은 시간과 비슷한 무게로 들어갑니다.
- 이제부터 이 0.51배를 출발점으로 두고, 조건을 더할 때 달라지는 정도를 봅니다.

### + 시간대

작은 예처럼 `C(hr)`를 더합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
m1 = smf.ols("np.log(cnt) ~ rain + C(hr)", data=h).fit()
np.exp(m1.params["rain"])
```

In [11]:
# ✍️ 위 코드를 여기에 직접 입력하세요
m1 = smf.ols("np.log(cnt) ~ rain + C(hr)", data=h).fit()
np.exp(m1.params["rain"])

np.float64(0.47777972423916426)

💬 **약 0.48배**로 조금 작아졌습니다. 비·눈은 대여가 많은 저녁 시간에 조금 더 많이 왔으므로(4-1 5절), 같은 시간대끼리 비교하면 비·눈의 차이가 조금 커집니다.

### + 기온·풍속

비·눈 시간은 평시보다 약 2도 춥고 바람이 셌습니다. 기온과 풍속은 크기에 뜻이 있는 숫자라 `C` 없이 그대로 `+`로 더합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
m2 = smf.ols("np.log(cnt) ~ rain + C(hr) + temp_c + windspeed", data=h).fit()
np.exp(m2.params["rain"])
```

In [12]:
# ✍️ 위 코드를 여기에 직접 입력하세요
m2 = smf.ols("np.log(cnt) ~ rain + C(hr) + temp_c + windspeed", data=h).fit()
np.exp(m2.params["rain"])

np.float64(0.5280391295484758)

💬 **약 0.53배**로 올라갔습니다. 비·눈 시간이 추웠던 몫을 맞추자, 비·눈 때문에 줄어드는 폭이 조금 작아졌습니다.

### + 월·근무일 여부·연도

이번에는 빈칸을 채워 직접 조건을 더합니다. 월 `mnth`는 시간대처럼 범주로, 근무일 여부 `workingday`와 연도 `yr`는 0·1 값 그대로 더합니다. 모형은 `m3`입니다.

In [ ]:
# ✍️ 빈칸(___)을 채운 뒤 실행하세요
m3 = smf.ols("np.log(cnt) ~ rain + C(hr) + temp_c + windspeed + ___", data=h).fit()
np.exp(m3.params["rain"])

✅ **체크포인트** — `0.5268...`이 나오면 맞습니다.

💬 **약 0.53배**로 거의 그대로입니다. 월·근무일 여부·연도가 비·눈의 배수에 섞인 몫은 작습니다.

지금까지 구한 배수를 맞춘 조건과 함께 한 표로 모읍니다.

In [22]:
# ▶ 그대로 실행하세요 — 맞춘 조건별 배수 비교표
means = h.groupby("rain")["cnt"].mean()
pd.DataFrame({"배수": [means.loc[1] / means.loc[0],
                       np.exp(m0.params["rain"]),
                       np.exp(m1.params["rain"]),
                       np.exp(m2.params["rain"]),
                       np.exp(m3.params["rain"])]},
             index=["평균끼리 (4-1)", "비·눈만 (m0)", "+ 시간대 (m1)",
                    "+ 기온·풍속 (m2)", "+ 월·근무일·연도 (m3)"]).round(3)

NameError: name 'm3' is not defined

💬 **결과 해설** — 평균끼리 0.57배 → 비·눈만 0.51배 → 시간대를 맞추면 0.48배 → 기온·풍속까지 0.53배 → 월·근무일·연도까지 0.53배입니다.
- 조건을 맞춰도 비·눈 시간의 대여는 **평시의 절반 남짓**입니다.
- 평균끼리의 배수가 크게 틀리지 않은 것은 비·눈이 시간대에 고르게 왔기 때문입니다(4-1 5절).
- 조건을 맞춰서 달라진 것은 주로 기온·풍속의 몫입니다. 둘 중 어느 쪽인지는 📝 과제 1에서 나눠 봅니다.

---
## 4. 넣지 않을 조건 — 습도

조건은 많이 넣을수록 좋을까요? 표에는 습도 `hum`도 있습니다. 조건으로 넣기 전에, 비·눈과 함께 달라지는 값인지 봅니다. 4-1의 **묶어서 평균** 패턴입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
h.groupby("rain")["hum"].mean().round(3)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 평균 습도는 평시 0.609, 비·눈 0.828입니다. 비나 눈이 오면 공기가 젖어 습도가 오릅니다. 습도는 비·눈과 따로 정해지는 조건이 아니라 **비·눈이 오면 따라 오르는 값**입니다.

그래도 습도까지 넣으면 배수가 어떻게 되는지 봅니다. `m3`의 식에 `hum`을 더해 `m4`에 담습니다.

In [ ]:
# ✍️ 빈칸(___)을 채운 뒤 실행하세요
m4 = smf.ols("np.log(cnt) ~ rain + C(hr) + temp_c + windspeed + C(mnth) + workingday + yr + ___", data=h).fit()
np.exp(m4.params["rain"])

💬 **결과 해설** — **약 0.56배**로, 비·눈의 차이가 작아 보입니다.
- 습도를 맞춘다는 것은 **습도가 같은 기록끼리** 비교한다는 뜻입니다.
- 비가 오면 습도가 오르므로, 습도가 같은 비·눈 시간과 평시를 비교하면 비 때문에 생긴 차이의 일부가 습도의 몫으로 넘어갑니다.
- 예보대로 비가 오면 습도도 함께 오릅니다. 계획에 필요한 것은 **비와 함께 오는 변화 전체**이므로 습도는 조건에서 빼고 `m3`(0.53배)를 씁니다.

---
## 5. 0.53배는 얼마나 흔들리나

0.53배는 2년 동안의 기록에서 나온 추정값이라 기록이 달랐다면 조금 다르게 나왔을 것입니다. 얼마나 흔들릴 수 있는지 `m3`의 rain 계수에 대한 95% 신뢰구간을 배수로 바꿔 봅니다. 구간은 두 가지로 계산합니다.

- **통상 구간**: 17,379시간을 모두 서로 독립인 기록으로 보고 계산합니다.
- **날짜로 묶은 구간**: 같은 날의 시간들을 한 묶음으로 보고 계산합니다. 같은 날의 비는 여러 시간 이어지고, 그날의 다른 사정도 그 시간들에 함께 영향을 줍니다. 이런 시간들을 서로 독립인 기록처럼 세면 구간이 실제보다 좁아집니다.

계산 코드는 이번 주에 따로 익히지 않으므로 실행만 합니다.

In [ ]:
# ▶ 그대로 실행하세요 — m3 배수의 95% 신뢰구간 (통상 구간, 날짜로 묶은 구간)
m3_day = smf.ols("np.log(cnt) ~ rain + C(hr) + temp_c + windspeed + C(mnth) + workingday + yr", data=h).fit(
    cov_type="cluster", cov_kwds={"groups": h["dteday"].factorize()[0]})
pd.DataFrame({"통상 구간": np.exp(m3.conf_int().loc["rain"]).values,
              "날짜로 묶은 구간": np.exp(m3_day.conf_int().loc["rain"]).values},
             index=["하한", "상한"]).round(3)

💬 **결과 해설** — 통상 구간은 0.509~0.546, 날짜로 묶은 구간은 0.486~0.571입니다.
- 날짜로 묶으면 구간이 넓어집니다. 계획에는 이 넓은 구간을 씁니다.
- 조건을 맞춘 배수는 0.53배이고, 기록의 흔들림을 감안한 범위는 **0.49~0.57배**입니다.

---
## 6. 근무일과 주말·휴일에 같은 배수를 써도 되나

근무일과 주말·휴일은 대여하는 사람과 목적이 다를 수 있어 비·눈에 다르게 반응할 수 있습니다. 기록을 둘로 나눠 각각 배수를 구합니다.

`h[조건]`은 조건이 참인 행만 남깁니다(1~3주차에서 쓴 조건 필터). 근무일(`workingday` 1)과 주말·휴일(`workingday` 0)의 기록을 나눕니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로)

```python
wd = h[h["workingday"] == 1]
we = h[h["workingday"] == 0]
len(wd), len(we)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 근무일 11,865시간, 주말·휴일 5,514시간입니다.

각 기록에 `m3`와 같은 조건으로 회귀합니다. 한쪽 기록 안에서는 `workingday`가 모두 같으므로 식에서 `workingday`만 뺍니다. 빈칸에 그 식을 씁니다.

In [ ]:
# ✍️ 빈칸(___)을 채운 뒤 실행하세요
m_wd = smf.ols(___, data=wd).fit()
m_we = smf.ols(___, data=we).fit()
np.exp(m_wd.params["rain"]), np.exp(m_we.params["rain"])

✅ **체크포인트** — `(0.5233..., 0.5333...)`이 나오면 맞습니다.

💬 근무일은 약 **0.52배**, 주말·휴일은 약 **0.53배**로 거의 같습니다. 근무일과 주말·휴일에 한 배수를 씁니다.

---
## 7. 결정

> **비·눈 예보가 있는 시간의 도시 전체 대여 계획은 그 시간의 평시 계획에 0.53을 곱해 잡는다.**

| 근거 | 값 |
|---|---|
| 조건을 맞춘 배수 (시간대·기온·풍속·월·근무일 여부·연도) | 0.53 |
| 날짜로 묶은 95% 신뢰구간 | 0.49~0.57 |
| 근무일 / 주말·휴일 | 0.52 / 0.53 |

이 배수는 실제로 비·눈이 기록된 시간의 비교이므로, 예보가 빗나간 시간은 들어 있지 않습니다. 강한 비·눈은 3시간뿐이라 약한 비·눈과 같은 배수로 묶였습니다.

---
## 8. 실습 과제 (15분)

### 📝 과제 1 · 기온과 풍속 중 무엇이 배수를 움직였나

3절에서 기온·풍속을 함께 더하자 0.48배가 0.53배로 올라갔습니다. `m1`의 식에 **기온만** 더한 모형과 **풍속만** 더한 모형의 배수를 각각 구하세요.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 3절의 m2 셀을 두 번 쓰고, 한 번은 식에서 windspeed를, 한 번은 temp_c를 뺍니다. 모형 이름은 m_temp, m_wind로 합니다.


✏️ **나의 답:** 0.48배를 0.53배로 올린 것은 주로 어느 쪽인가요?

### 📝 과제 2 · 로그를 씌우지 않으면?

`m3`와 같은 조건으로, 결과에 로그를 씌우지 않고 `cnt` 그대로 회귀해 보세요. 이때 rain의 계수는 배수가 아니라 **건수의 차이**이므로 `np.exp`를 씌우지 않고 계수만 봅니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 3절의 m3 식에서 np.log(cnt)를 cnt로 바꾸고, 모형 이름은 m_lin으로 합니다. 마지막 줄은 np.exp 없이 계수만 봅니다.


✏️ **나의 답:** 계수는 몇 건인가요? 4-1에서 새벽 4시의 평시 평균은 6.5건, 오후 5시는 483.0건이었습니다. 두 시간대에서 이 건수를 똑같이 빼면 어떻게 되나요?

### 📝 과제 3 · 생각해 보기

습도는 비가 오면 따라 오르는 값이라 조건에서 뺐습니다. 기온과 풍속은 어떤가요? 비가 오면 함께 달라지는 값일 수 있을까요? 그렇다면 기온·풍속을 조건에 넣은 0.53배는 어느 쪽으로 치우쳤을 수 있을까요?

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 로그와 배수 | 로그 차이에 `np.exp`를 씌우면 배수. 결과에 로그를 씌운 회귀의 계수 → `np.exp` → 배수 |
| 조건을 맞춘다 | 그 조건이 같은 기록끼리 비교한다. 작은 예에서 비만 넣으면 0.31배, 시간대를 맞추면 0.5배 |
| 조건을 하나씩 더하기 | 평균끼리 0.57 → 비·눈만 0.51 → +시간대 0.48 → +기온·풍속 0.53 → +월·근무일·연도 0.53 |
| 넣지 않을 조건 | 습도는 비가 오면 따라 오른다. 넣으면 0.56배로 비의 차이가 작아 보인다 |
| 흔들림 | 날짜로 묶은 95% 신뢰구간 0.49~0.57 |
| 결정 | 비·눈 예보 시간의 대여 계획 = 평시 계획 × **0.53**. 근무일 0.52, 주말·휴일 0.53으로 한 값을 쓴다 |
| 반복한 패턴 | `smf.ols("np.log(cnt) ~ rain + 조건", data=h).fit()` → `np.exp(모형.params["rain"])` |

### 이번 주에 할 일

| 활동 | 자료 | 비고 |
|---|---|---|
| 배경지식 | 4주차 배경지식 문서 | 회귀계수의 뜻, 범주 변수, 로그와 배수, 넣지 않을 조건 |
| 교재 | 4-1, 4-2 노트북 | 평균으로 비교한 뒤 조건을 하나씩 맞춰 가며 배수를 읽는다 |
| 랩 | `week04_lab.ipynb` — 같은 공유자전거 기록 | 회원과 비회원은 비·눈에 같은 배수로 반응할까? 두 집단의 배수를 따로 구한다. 약 60분 |